
# Experiment 02 — FedAvg under Pathological Label Skew (MNIST)

This notebook is the **complete, reproducible version of Experiment 02**.

It isolates one question:

> **How does FedAvg behave when each client has a highly concentrated label distribution, and how does the number of local epochs $E$ change the communication–computation trade-off?**

The reusable implementation stays in `src/`. This notebook changes only the client data partition and experiment configuration.

### Controlled variables

Across the IID baseline and this experiment we keep fixed:

- MNIST preprocessing;
- 5 clients;
- `SimpleMLP(784 → 128 → 10)`;
- SGD with learning rate $0.01$;
- batch size $64$;
- the same test set;
- the same saved initial checkpoint $w_0$;
- full client participation ($C=1$);
- 20 communication rounds.

The only intended change from Experiment 01 is the **training-data distribution across clients**.

### Completed runs

The final shard experiment contains:

$$
E \in \{1,5,10\},\qquad T=20.
$$

The $E=1$ and $E=5$ histories are the original saved CSVs.  
The $E=10$ history was reconstructed from the retained round-by-round console log after the compute session expired. Its logged loss/accuracy values are therefore rounded to four decimal places, and its runtime is intentionally not fabricated.


## 1. Setup

In [ ]:

from pathlib import Path
import os
import sys

def find_repo_root():
    """Find the repository root from local VS Code, Kaggle, or Colab."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    # Optional remote-compute locations.
    candidates += [
        Path("/kaggle/working/federated-learning-under-heterogeneity"),
        Path("/content/federated-learning-under-heterogeneity"),
    ]

    for candidate in candidates:
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate.resolve()

    raise RuntimeError(
        "Repository root not found. Open the "
        "'federated-learning-under-heterogeneity' folder in VS Code "
        "or clone/cd into the repository before running this notebook."
    )

REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root  :", REPO_ROOT)
print("Working dir:", Path.cwd())
print("Python     :", sys.executable)


In [ ]:

import copy
import random
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from src.models import SimpleMLP
from src.data import (
    partition_shard,
    create_client_loaders,
    client_label_counts,
)
from src.training import (
    evaluate,
    client_update,
    federated_train,
)
from src.metrics import model_distance

print("Imports successful.")


In [ ]:

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_best_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_mnist(root):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    train_ds = datasets.MNIST(
        root=root,
        train=True,
        download=True,
        transform=transform,
    )
    test_ds = datasets.MNIST(
        root=root,
        train=False,
        download=True,
        transform=transform,
    )
    return train_ds, test_ds


def verify_partition(client_indices, dataset_size: int):
    flat = [
        int(idx)
        for indices in client_indices.values()
        for idx in indices
    ]

    checks = {
        "dataset_size": dataset_size,
        "assigned": len(flat),
        "unique": len(set(flat)),
        "all_clients_nonempty": all(len(v) > 0 for v in client_indices.values()),
    }

    if checks["assigned"] != dataset_size:
        raise ValueError("Partition lost or added examples.")
    if checks["unique"] != dataset_size:
        raise ValueError("Partition contains duplicate or missing indices.")
    if not checks["all_clients_nonempty"]:
        raise ValueError("At least one client is empty.")

    return checks


def load_torch_object(path, map_location="cpu"):
    """Compatibility helper across PyTorch versions."""
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


In [ ]:

@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    num_clients: int = 5
    shards_per_client: int = 2
    batch_size: int = 64
    learning_rate: float = 0.01
    num_rounds: int = 20
    local_epoch_values: tuple = (1, 5, 10)

    datasets_dir: str = "datasets"
    iid_results_dir: str = "results/iid_baseline"
    results_dir: str = "results/shard_noniid"


CFG = ExperimentConfig()
RESULT_DIR = REPO_ROOT / CFG.results_dir
RESULT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_DIR = RESULT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

device = get_best_device()

print(CFG)
print("Device:", device)
print("Results:", RESULT_DIR)



## 2. Load MNIST

We use exactly the same preprocessing as the IID baseline:

$$
x \mapsto \frac{x-0.5}{0.5}.
$$

Changing preprocessing here would confound the comparison.


In [ ]:

set_seed(CFG.seed)

train_ds, test_ds = load_mnist(REPO_ROOT / CFG.datasets_dir)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
)

print("Training examples:", len(train_ds))
print("Test examples    :", len(test_ds))



## 3. Create or load the fixed pathological-shard partition

The 60,000 training examples are sorted by label and divided into

$$
K\times s=5\times2=10
$$

shards.

Each shard contains 6,000 examples and each client receives two shards, so every client has 12,000 examples.

Because shard boundaries are formed after sorting by label, each local dataset is dominated by only a few digits. This is intentionally **severe, pathological label skew**.

For exact reproducibility, if `results/shard_noniid/client_indices.pt` exists, the notebook loads that previously saved partition instead of silently generating a new one.


In [ ]:

PARTITION_PATH = RESULT_DIR / "client_indices.pt"

if PARTITION_PATH.exists():
    client_indices = load_torch_object(PARTITION_PATH)
    print("Loaded saved partition:", PARTITION_PATH)
else:
    client_indices = partition_shard(
        train_ds,
        num_clients=CFG.num_clients,
        seed=CFG.seed,
        shard_per_client=CFG.shards_per_client,
    )
    torch.save(client_indices, PARTITION_PATH)
    print("Created and saved partition:", PARTITION_PATH)

partition_check = verify_partition(client_indices, len(train_ds))

client_loaders = create_client_loaders(
    train_ds,
    client_indices,
    batch_size=CFG.batch_size,
)

print(partition_check)
print("Client sizes:")
print({client_id: len(indices) for client_id, indices in client_indices.items()})


## 4. Inspect and save the client label distributions

In [ ]:

distribution_df = client_label_counts(train_ds, client_indices)

distribution_path = RESULT_DIR / "client_label_distribution.csv"
distribution_df.to_csv(distribution_path, index=False)

distribution_df


In [ ]:

digit_columns = [f"digit_{d}" for d in range(10)]

label_proportions = (
    distribution_df[digit_columns]
    .div(distribution_df["samples"], axis=0)
)

top2_share = np.sort(
    label_proportions.to_numpy(),
    axis=1,
)[:, -2:].sum(axis=1)

concentration_df = pd.DataFrame({
    "client": distribution_df["client"],
    "top_2_label_share": top2_share,
})

concentration_df


In [ ]:

fig, ax = plt.subplots(figsize=(9, 4))

image = ax.imshow(
    label_proportions.to_numpy(),
    aspect="auto",
    vmin=0,
    vmax=1,
)

ax.set_xticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticks(range(CFG.num_clients))
ax.set_yticklabels([f"Client {i}" for i in range(CFG.num_clients)])
ax.set_xlabel("Digit label")
ax.set_ylabel("Client")
ax.set_title("Client label proportions — pathological shard split")

fig.colorbar(image, ax=ax, label="Proportion of client data")
plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "client_label_proportions.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()



### Validated partition for seed 42

The saved partition is strongly concentrated:

- **Client 0:** mostly digits 2 and 6;
- **Client 1:** mostly digits 1 and 8;
- **Client 2:** mostly digits 4 and 5;
- **Client 3:** mostly digits 0 and 9;
- **Client 4:** mostly digits 3 and 7.

Small counts from neighboring digits are expected because a fixed 6,000-example shard boundary does not exactly coincide with each MNIST class boundary.

The important invariants are:

$$
\text{assigned}=60{,}000,\qquad
\text{unique}=60{,}000.
$$

Therefore every training example is assigned exactly once.


## 5. Load and verify the common initial checkpoint

In [ ]:

INITIAL_MODEL_PATH = REPO_ROOT / CFG.iid_results_dir / "initial_model.pt"

if not INITIAL_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing common initial checkpoint: {INITIAL_MODEL_PATH}\n"
        "Experiment 02 must use the same w0 as the IID baseline."
    )

initial_state = load_torch_object(INITIAL_MODEL_PATH)

initial_model = SimpleMLP(784, 10)
initial_model.load_state_dict(initial_state)
initial_model.to(device)

loss_fn = nn.CrossEntropyLoss()
initial_loss, initial_acc = evaluate(
    initial_model,
    test_loader,
    loss_fn,
    device,
)

print(f"Initial loss    : {initial_loss:.6f}")
print(f"Initial accuracy: {initial_acc:.4f}")

# Loose tolerances catch the wrong checkpoint/preprocessing without being brittle.
assert abs(initial_loss - 2.313339) < 1e-3
assert abs(initial_acc - 0.1267) < 1e-4



The expected untouched-model evaluation is approximately:

$$
\text{loss}=2.313339,\qquad
\text{accuracy}=12.67\%.
$$

Matching this value is an important experimental-control check: changing the client partition must not change the initial global model or test evaluation.


## 6. Optional client-update diagnostic

In [ ]:

RUN_CLIENT_DIAGNOSTIC = False

if RUN_CLIENT_DIAGNOSTIC:
    diagnostic_global = SimpleMLP(784, 10)
    diagnostic_global.load_state_dict(copy.deepcopy(initial_state))
    diagnostic_global.to(device)

    global_before = copy.deepcopy(diagnostic_global)

    local_model = client_update(
        diagnostic_global,
        client_loaders[0],
        local_epochs=1,
        learning_rate=CFG.learning_rate,
        loss_fn=loss_fn,
        device=device,
    )

    print(
        "Local movement:",
        model_distance(diagnostic_global, local_model),
    )
    print(
        "Global changed:",
        model_distance(global_before, diagnostic_global),
    )
else:
    print("Skipped. Set RUN_CLIENT_DIAGNOSTIC=True to reproduce it.")



A previous validation gave local movement of approximately $1.1444$ for Client 0 at $E=1$, while the server model changed by $0.0$ inside `client_update`.

This confirms the intended contract: local training modifies a copied client model, not the current global model.


## 7. Experiment runner

In [ ]:

def run_experiment(
    local_epochs: int,
    num_rounds: int = CFG.num_rounds,
    verbose: bool = True,
):
    """Run FedAvg from the exact common initial state."""
    set_seed(CFG.seed)

    model = SimpleMLP(784, 10)
    model.load_state_dict(copy.deepcopy(initial_state))
    model.to(device)

    start = time.perf_counter()

    trained_model, history = federated_train(
        global_model=model,
        client_loaders=client_loaders,
        num_rounds=num_rounds,
        local_epochs=local_epochs,
        learning_rate=CFG.learning_rate,
        loss_fn=nn.CrossEntropyLoss(),
        test_loader=test_loader,
        device=device,
        verbose=verbose,
    )

    runtime = time.perf_counter() - start
    return trained_model, history, runtime


def history_path(E: int):
    return RESULT_DIR / f"fedavg_shard_E{E}_history.csv"


def validate_history(history: pd.DataFrame, E: int):
    required = {"round", "test_loss", "test_accuracy"}
    missing = required.difference(history.columns)

    if missing:
        raise ValueError(f"E={E}: missing columns {missing}")

    expected_rounds = list(range(CFG.num_rounds + 1))
    actual_rounds = history["round"].astype(int).tolist()

    if actual_rounds != expected_rounds:
        raise ValueError(
            f"E={E}: expected rounds 0..{CFG.num_rounds}, "
            f"got {actual_rounds}"
        )

    if abs(float(history.iloc[0]["test_loss"]) - 2.313339) > 1e-3:
        raise ValueError(f"E={E}: round-0 loss does not match w0.")

    if abs(float(history.iloc[0]["test_accuracy"]) - 0.1267) > 1e-4:
        raise ValueError(f"E={E}: round-0 accuracy does not match w0.")

    return history



## 8. Load the completed $E$-sweep

The expensive experiment has already been completed for

$$
E\in\{1,5,10\},\qquad T=20.
$$

Therefore this notebook **loads saved histories by default** instead of wasting compute.

If a history is genuinely missing, set `RUN_MISSING_EXPERIMENTS=True` to reproduce only that missing run.

> **Provenance note for $E=10$.** The compute session expired before the CSV could be saved. The complete round-by-round console output was retained, so the history was reconstructed from that log. The logged values have four-decimal precision. The runtime was not recoverable and is not invented.


In [ ]:

RUN_MISSING_EXPERIMENTS = False

histories = {}
runtimes = {}

for E in CFG.local_epoch_values:
    path = history_path(E)

    if path.exists():
        history = pd.read_csv(path)
        histories[E] = validate_history(history, E)
        print(f"E={E}: loaded {path.name}")
        continue

    if not RUN_MISSING_EXPERIMENTS:
        raise FileNotFoundError(
            f"Missing {path}.\n"
            "Restore the saved result file, or set "
            "RUN_MISSING_EXPERIMENTS=True to rerun it."
        )

    print(f"E={E}: result missing; running experiment...")
    _, history, runtime = run_experiment(
        local_epochs=E,
        num_rounds=CFG.num_rounds,
        verbose=True,
    )

    history = validate_history(history, E)
    history.to_csv(path, index=False)

    histories[E] = history
    runtimes[E] = runtime

    print(f"E={E}: saved {path.name}")


In [ ]:

for E in CFG.local_epoch_values:
    print(f"\nE={E}")
    display(histories[E].tail())


## 9. Summarize the completed experiment

In [ ]:

def first_round_at_accuracy(history, threshold):
    hit = history.loc[
        history["test_accuracy"] >= threshold,
        "round",
    ]
    return int(hit.iloc[0]) if len(hit) else None


summary_rows = []

for E in CFG.local_epoch_values:
    history = histories[E]

    r70 = first_round_at_accuracy(history, 0.70)
    r75 = first_round_at_accuracy(history, 0.75)
    r80 = first_round_at_accuracy(history, 0.80)

    summary_rows.append({
        "local_epochs": E,
        "final_loss": float(history.iloc[-1]["test_loss"]),
        "final_accuracy": float(history.iloc[-1]["test_accuracy"]),
        "round_to_70": r70,
        "round_to_75": r75,
        "round_to_80": r80,
        "cumulative_local_epochs_to_70": r70 * E if r70 is not None else np.nan,
        "cumulative_local_epochs_to_75": r75 * E if r75 is not None else np.nan,
        "cumulative_local_epochs_to_80": r80 * E if r80 is not None else np.nan,
    })


summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULT_DIR / "summary.csv", index=False)

summary_df



### Observed result

| Local epochs $E$ | Final loss | Final accuracy | Round to 70% | Round to 75% | Round to 80% |
|---:|---:|---:|---:|---:|---:|
| 1 | 0.6301 | 78.84% | 10 | 14 | not reached |
| 5 | 0.5471 | 79.98% | 7 | 10 | not reached |
| 10 | 0.4829 | **82.43%** | 7 | 9 | 15 |

Increasing $E$ improves progress **per communication round**, but the amount of local computation rises much faster.

For example, reaching about 75% accuracy requires approximately:

$$
\begin{aligned}
E=1 &: 14\times1=14 \text{ cumulative local epochs/client},\\
E=5 &: 10\times5=50,\\
E=10 &: 9\times10=90.
\end{aligned}
$$

So increasing $E$ from 5 to 10 saves only about one communication round at this threshold, while cumulative local computation increases from 50 to 90 epochs/client.


## 10. Accuracy vs communication rounds

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))

for E in CFG.local_epoch_values:
    history = histories[E]
    ax.plot(
        history["round"],
        history["test_accuracy"],
        marker="o",
        markersize=3,
        label=f"E={E}",
    )

ax.set_xlabel("Communication round")
ax.set_ylabel("Test accuracy")
ax.set_title("FedAvg under pathological label skew")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(
    FIGURE_DIR / "accuracy_vs_round.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()



This view measures **communication efficiency**.

Larger $E$ means clients perform more local optimization before each aggregation. Therefore $E=5$ and $E=10$ generally reach a given accuracy in fewer communication rounds than $E=1$.


## 11. Loss vs communication rounds

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))

for E in CFG.local_epoch_values:
    history = histories[E]
    ax.plot(
        history["round"],
        history["test_loss"],
        marker="o",
        markersize=3,
        label=f"E={E}",
    )

ax.set_xlabel("Communication round")
ax.set_ylabel("Test loss")
ax.set_title("Test loss under pathological label skew")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(
    FIGURE_DIR / "loss_vs_round.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()


## 12. Accuracy vs cumulative local computation

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))

for E in CFG.local_epoch_values:
    history = histories[E]

    cumulative_local_epochs = history["round"] * E

    ax.plot(
        cumulative_local_epochs,
        history["test_accuracy"],
        marker="o",
        markersize=3,
        label=f"E={E}",
    )

ax.set_xlabel("Cumulative local epochs per client")
ax.set_ylabel("Test accuracy")
ax.set_title("Compute efficiency under pathological label skew")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(
    FIGURE_DIR / "accuracy_vs_cumulative_local_epochs.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()



This view answers a different question from accuracy-vs-round:

> **How much global accuracy do we obtain for the amount of local optimization performed?**

The experiment shows a clear communication–computation trade-off:

- larger $E$ improves communication efficiency;
- however, the gain shows diminishing returns;
- severe label heterogeneity is not solved merely by performing more local SGD.

The data do **not** justify the claim that larger $E$ is always harmful under non-IID data. In this experiment, $E=10$ has the best round-20 accuracy. The supported conclusion is narrower: **more local computation continues to help per round, but becomes substantially less efficient when measured by cumulative local work.**


## 13. Optional comparison with the IID baseline

In [ ]:

def find_iid_history(E: int):
    iid_dir = REPO_ROOT / CFG.iid_results_dir

    candidates = [
        iid_dir / f"fedavg_iid_E{E}_history.csv",
        iid_dir / f"iid_E{E}_history.csv",
    ]

    for candidate in candidates:
        if candidate.exists():
            return candidate

    matches = sorted(iid_dir.glob(f"*E{E}*history*.csv"))
    return matches[0] if matches else None


iid_histories = {}
for E in CFG.local_epoch_values:
    path = find_iid_history(E)
    if path is not None:
        iid_histories[E] = pd.read_csv(path)
        print(f"IID E={E}: loaded {path.name}")
    else:
        print(f"IID E={E}: history not found; skipping.")


if len(iid_histories) == len(CFG.local_epoch_values):
    comparison_rows = []

    for E in CFG.local_epoch_values:
        iid_final = float(iid_histories[E].iloc[-1]["test_accuracy"])
        shard_final = float(histories[E].iloc[-1]["test_accuracy"])

        comparison_rows.append({
            "local_epochs": E,
            "iid_final_accuracy": iid_final,
            "shard_final_accuracy": shard_final,
            "accuracy_gap_iid_minus_shard": iid_final - shard_final,
        })

    iid_vs_shard_df = pd.DataFrame(comparison_rows)
    display(iid_vs_shard_df)

    iid_vs_shard_df.to_csv(
        RESULT_DIR / "iid_vs_shard_summary.csv",
        index=False,
    )
else:
    print(
        "Not all IID histories are present. "
        "The shard experiment itself is still complete."
    )



If the IID histories from Experiment 01 are available, this section quantifies the accuracy gap under identical $E$, model, initialization, learning rate, batch size, rounds, and test set.

That is the clean comparison needed to attribute the degradation primarily to the pathological client label distribution.


## 14. Final conclusions


### What Experiment 02 establishes

1. **The pathological shard split is valid and severe.**  
   Every MNIST training example is assigned exactly once, each client contains 12,000 examples, and each local label distribution is dominated by a small number of digits.

2. **FedAvg still learns, but severe statistical heterogeneity strongly limits its global performance.**  
   After 20 communication rounds, the observed shard accuracies are 78.84%, 79.98%, and 82.43% for $E=1,5,10$.

3. **Increasing $E$ improves communication efficiency.**  
   Higher $E$ generally reaches accuracy thresholds in fewer rounds.

4. **The communication gains have diminishing returns when local computation is counted.**  
   Roughly 75% accuracy is reached after about 14, 50, and 90 cumulative local epochs/client for $E=1,5,10$, respectively.

5. **We should not yet claim that client drift has been causally demonstrated.**  
   The experiment demonstrates the effect of severe statistical heterogeneity and an $E$-dependent communication–computation trade-off. Update-direction diagnostics or a mitigation method such as FedProx are needed for a stronger client-drift argument.

### Next experiment

Experiment 03 replaces the artificial shard split with **Dirichlet label skew**, providing a tunable heterogeneity parameter $\alpha$. This lets us move from a binary IID-vs-pathological comparison to a controlled heterogeneity axis.


## 15. Artifact check

In [ ]:

expected_files = [
    RESULT_DIR / "client_indices.pt",
    RESULT_DIR / "client_label_distribution.csv",
    RESULT_DIR / "fedavg_shard_E1_history.csv",
    RESULT_DIR / "fedavg_shard_E5_history.csv",
    RESULT_DIR / "fedavg_shard_E10_history.csv",
    RESULT_DIR / "summary.csv",
]

artifact_check = pd.DataFrame({
    "artifact": [str(path.relative_to(REPO_ROOT)) for path in expected_files],
    "exists": [path.exists() for path in expected_files],
})

artifact_check
